# Run_BrickScanner

Purpose: Ad hoc BrickScanner execution with configurable parameters.

**Widgets:**
- `include_paths`: Comma-separated workspace paths to scan
- `dry_run`: Whether to run in dry-run mode (returns results without persisting)
- `run_setup`: Whether to run DB_Setup before scanning

## Define Widgets

In [ ]:
# Define widgets
dbutils.widgets.text("include_paths", "/Workspace", "Include Paths (comma-separated)")
dbutils.widgets.checkbox("dry_run", False, "Dry Run (returns results, no persist)")
dbutils.widgets.checkbox("run_setup", False, "Run DB_Setup first")

include_paths = dbutils.widgets.get("include_paths")
dry_run = dbutils.widgets.get("dry_run").lower() == "true"
run_setup = dbutils.widgets.get("run_setup").lower() == "true"

## Install Dependencies

In [ ]:
import sys
import subprocess

print("[Run_BrickScanner] Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "/dbfs/user/sai_kalluri@bcbsil.com/CLOUD_BIS_AI_POC/BrickScanner/requirements.txt"])

# Restart Python to ensure clean imports
dbutils.library.restartPython()

## Setup Imports and Configuration

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("[Run_BrickScanner]")

# Add project to path
sys.path.insert(0, "/dbfs/user/sai_kalluri@bcbsil.com/CLOUD_BIS_AI_POC/BrickScanner")

## Validate Inputs

In [ ]:
# Validate inputs
if not include_paths or not include_paths.strip():
    raise ValueError("include_paths cannot be empty")

print(f"[Run_BrickScanner] Configuration:")
print(f"  Include Paths: {include_paths}")
print(f"  Dry Run: {dry_run}")
print(f"  Run Setup: {run_setup}")

## Run DB_Setup if Requested

In [ ]:
# Run setup if requested
if run_setup:
    print("[Run_BrickScanner] Running DB_Setup...")
    dbutils.notebook.run("./DB_Setup.ipynb")
    print("[Run_BrickScanner] DB_Setup completed")

## Execute Scanner

In [ ]:
# Import and execute scanner
print("[Run_BrickScanner] Importing BrickScanner package...")
import brickscanner

print("[Run_BrickScanner] Starting scan...")
try:
    result_df = brickscanner.run(
        spark,
        extractor_type="genai",
        include_paths=include_paths,
        dry_run=dry_run
    )
    
    if dry_run:
        print("[Run_BrickScanner] ✓ Dry run completed")
        display(result_df)
    else:
        print("[Run_BrickScanner] ✓ Scan completed and results persisted")
        
        # Query and display summary
        from brickscanner.config import DB_TABLES
        output_table = DB_TABLES["genai_output"]
        
        summary_df = spark.sql(f"""
            SELECT 
                severity,
                COUNT(*) as count
            FROM {output_table}
            GROUP BY severity
            ORDER BY 
                CASE 
                    WHEN severity = '1-High' THEN 1
                    WHEN severity = '2-Medium' THEN 2
                    WHEN severity = '3-Low' THEN 3
                    ELSE 4
                END
        """)
        
        print("[Run_BrickScanner] Summary:")
        display(summary_df)
        
        total = spark.sql(f"SELECT COUNT(*) as total FROM {output_table}").collect()[0].total
        print(f"[Run_BrickScanner] Total findings: {total}")

except Exception as e:
    logger.error(f"Scan failed: {e}")
    raise